# Day03：淘宝商品数据Pandas探索

**姓名：** 请填写

本Notebook由每名学生独立完成，并提交到个人GitHub仓库。

> 请完成所有TODO和检查点。不要覆盖原始数据文件。

## 实验目标

你需要完成：

1. 读取25,000条淘宝商品记录；
2. 检查字段类型和缺失值；
3. 完成列选择、行选择、条件筛选和排序；
4. 完成商品价格及一级品类统计；
5. 完成“省份—类别”挑战分析；
6. 输出两张统计表并撰写有边界的结论。

### 数据边界

- 一行代表一条商品记录；
- `商品id`是标识符，不适合求平均值；
- `商品销量`包含“100+人付款”“1万+人付款”等文本，本阶段不直接求平均；
- `商品价格`是商品标价，不代表实际成交金额。

## 任务0：环境与个人配置

In [29]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


STUDENT_NAME = "王仁献"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "淘宝全品类全国数据.csv").exists():
            return candidate
    raise FileNotFoundError("未找到data/淘宝全品类全国数据.csv")


ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "淘宝全品类全国数据.csv"
OUTPUT_DIR = ROOT / "output" / "day03_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


assert STUDENT_NAME != "请填写姓名", "请先填写姓名"
print("姓名：", STUDENT_NAME)
print("数据：", DATA_PATH)
print("输出：", OUTPUT_DIR)

姓名： 王仁献
数据： d:\training\data\淘宝全品类全国数据.csv
输出： d:\training\output\day03_analysis


## 任务1：读取数据并完成初步观察

In [30]:
df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
print("columns:", df.columns.tolist())
print("\n数据统计预览：")
display(df.head(5))
df.info()
display(df.describe(include="all"))

shape: (25000, 15)
columns: ['商品id', '一级品类', '二级品类', '商品标题', '商品价格', '省份', '商品销量', '店铺名称', '店铺标签', '先用后付', '退货宝', '风格', '面料', '版型', '适用季节']

数据统计预览：


,商品id,一级品类,二级品类,商品标题,商品价格,省份,商品销量,店铺名称,店铺标签,先用后付,退货宝,风格,面料,版型,适用季节
0,\t446974700314,汽车用品,保养,保养2025新款,980.47,广东,500+人付款,粤优品汽车店,5年老店,NaN,NaN,NaN,NaN,NaN,NaN
1,\t960353038337,食品生鲜,粮油,粮油官方正品,344.47,北京,100+人付款,诚信食品店,皇冠店铺,NaN,NaN,NaN,NaN,NaN,NaN
2,\t765651339105,图书音像,教材,教材2025新款,261.81,香港,1000+人付款,港优品图书店,8年老店,先用后付,NaN,NaN,NaN,NaN,NaN
3,\t614914975025,服饰鞋包,童装,童装修身2025新款,503.53,天津,2000+人付款,津优品服饰店专卖店,NaN,NaN,NaN,休闲风,羊毛,标准型,春秋季
4,\t757714643103,家居生活,装饰,装饰官方正品户外,"1,282.75",北京,500+人付款,时尚家居店旗舰店,回头客3千,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   商品id    25000 non-null  object 
 1   一级品类    25000 non-null  object 
 2   二级品类    25000 non-null  object 
 3   商品标题    25000 non-null  object 
 4   商品价格    25000 non-null  float64
 5   省份      25000 non-null  object 
 6   商品销量    25000 non-null  object 
 7   店铺名称    25000 non-null  object 
 8   店铺标签    21259 non-null  object 
 9   先用后付    3158 non-null   object 
 10  退货宝     2724 non-null   object 
 11  风格      3988 non-null   object 
 12  面料      2375 non-null   object 
 13  版型      2036 non-null   object 
 14  适用季节    4822 non-null   object 
dtypes: float64(1), object(14)
memory usage: 2.9+ MB


,商品id,一级品类,二级品类,商品标题,商品价格,省份,商品销量,店铺名称,店铺标签,先用后付,退货宝,风格,面料,版型,适用季节
count,25000,25000,25000,25000,"25,000.00",25000,25000,25000,21259,3158,2724,3988,2375,2036,4822
unique,25000,15,71,1029,NaN,34,13,3810,10,1,1,3,5,3,5
top,\t446974700314,数码家电,玩具,保健品2025新款,NaN,广东,1000+人付款,品质运动店,5年老店,先用后付,退货宝,休闲风,羊毛,宽松型,夏季
freq,1,1712,683,160,NaN,2303,4378,172,2206,3158,2724,1356,514,716,999
mean,NaN,NaN,NaN,NaN,938.26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,"1,017.92",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,11.26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,257.39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,617.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,"1,211.89",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# 检查点1
assert df.shape == (25000, 15), "数据形状应为(25000, 15)"
assert {"商品id", "一级品类", "商品价格", "省份", "商品销量"}.issubset(df.columns)
print("检查点1通过")

检查点1通过


**数据粒度：** 请用一句话说明一行代表什么。

> 一行代表一个淘宝商品的商品记录，包含该商品的id、类目、价格、销量、省份等信息。

## 任务2：字段类型与缺失值

In [32]:
# TODO 1：输出字段类型
print(df.dtypes)

# TODO 2：计算缺失数量并从高到低排序，变量名为missing_count
missing_count = df.isna().sum().sort_values(ascending=False)

# TODO 3：计算缺失率（百分比），变量名为missing_rate
missing_rate = (missing_count / len(df) * 100).round(2)

# TODO 4：展示结果
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_rate": missing_rate,
})
print("\n缺失值统计：")
display(missing_summary)

商品id     object
一级品类     object
二级品类     object
商品标题     object
商品价格    float64
省份       object
商品销量     object
店铺名称     object
店铺标签     object
先用后付     object
退货宝      object
风格       object
面料       object
版型       object
适用季节     object
dtype: object

缺失值统计：


,missing_count,missing_rate
版型,22964,91.86
面料,22625,90.50
退货宝,22276,89.10
先用后付,21842,87.37
风格,21012,84.05
适用季节,20178,80.71
店铺标签,3741,14.96
商品id,0,0.00
一级品类,0,0.00
二级品类,0,0.00


In [33]:
# 检查点2
assert isinstance(missing_count, pd.Series), "missing_count应为Series"
assert isinstance(missing_rate, pd.Series), "missing_rate应为Series"
assert set(missing_count.index) == set(df.columns)
assert missing_count.sum() == df.isna().sum().sum()
print("检查点2通过")

检查点2通过


1.	可直接数值统计字段：商品价格
原因：浮点数值类型，无缺失，可直接计算均值、中位数、最值等统计指标。
2.	暂不宜直接精确数值统计字段：商品销量
原因：字段为字符串文本（如500+人付款、2万+人付款），包含文字符号，无法直接参与数学运算，需后续清洗转换。


## 任务3：选择列与选择行

In [34]:
# TODO 1：选择“商品价格”列，变量名为price_series
price_series = df["商品价格"]

# TODO 2：选择商品id、一级品类、商品价格、省份、商品销量五列
product_view = df[["商品id", "一级品类", "商品价格", "省份", "商品销量"]]

# TODO 3：分别使用loc和iloc取前5行局部数据
loc_view = df.loc[:4, ["商品id", "一级品类", "商品价格", "省份", "商品销量"]]
iloc_view = df.iloc[:5, [0, 1, 4, 5, 6]]

# TODO 4：展示结果
print(price_series.head())
print(product_view.head())
print(loc_view)
print(iloc_view)

0     980.47
1     344.47
2     261.81
3     503.53
4   1,282.75
Name: 商品价格, dtype: float64
             商品id  一级品类     商品价格  省份      商品销量
0  \t446974700314  汽车用品   980.47  广东   500+人付款
1  \t960353038337  食品生鲜   344.47  北京   100+人付款
2  \t765651339105  图书音像   261.81  香港  1000+人付款
3  \t614914975025  服饰鞋包   503.53  天津  2000+人付款
4  \t757714643103  家居生活 1,282.75  北京   500+人付款
             商品id  一级品类     商品价格  省份      商品销量
0  \t446974700314  汽车用品   980.47  广东   500+人付款
1  \t960353038337  食品生鲜   344.47  北京   100+人付款
2  \t765651339105  图书音像   261.81  香港  1000+人付款
3  \t614914975025  服饰鞋包   503.53  天津  2000+人付款
4  \t757714643103  家居生活 1,282.75  北京   500+人付款
             商品id  一级品类     商品价格  省份      商品销量
0  \t446974700314  汽车用品   980.47  广东   500+人付款
1  \t960353038337  食品生鲜   344.47  北京   100+人付款
2  \t765651339105  图书音像   261.81  香港  1000+人付款
3  \t614914975025  服饰鞋包   503.53  天津  2000+人付款
4  \t757714643103  家居生活 1,282.75  北京   500+人付款


In [35]:
# 检查点3
assert isinstance(price_series, pd.Series)
assert isinstance(product_view, pd.DataFrame)
assert product_view.shape == (25000, 5)
assert len(loc_view) == 5 and len(iloc_view) == 5
print("检查点3通过")

检查点3通过


df["商品价格"] 返回Series 一维序列，只有一列数据，无列名结构；
df[["商品价格"]] 返回DataFrame 二维表格，保留表格行列结构，支持多列拓展操作。
请解释`df["商品价格"]`与`df[["商品价格"]]`的区别。


## 任务4：条件筛选与排序

In [36]:
# TODO 1：筛选广东商品，变量名为guangdong
guangdong = df[df["省份"] == "广东"].copy()

# TODO 2：筛选广东且商品价格不低于1000元的商品
# 只保留商品id、一级品类、二级品类、商品价格、省份、商品销量
# 按商品价格从高到低排序，变量名为guangdong_high_price
guangdong_high_price = guangdong[guangdong["商品价格"] >= 1000][
    ["商品id", "一级品类", "二级品类", "商品价格", "省份", "商品销量"]
].sort_values("商品价格", ascending=False).reset_index(drop=True)

# TODO 3：筛选浙江或江苏商品，变量名为zhejiang_or_jiangsu
zhejiang_or_jiangsu = df[df["省份"].isin(["浙江", "江苏"])].copy()

# TODO 4：展示广东高价商品前10行和浙江或江苏商品数
print("广东高价商品前10行：")
display(guangdong_high_price.head(10))
print("浙江或江苏商品数量：", len(zhejiang_or_jiangsu))

广东高价商品前10行：


,商品id,一级品类,二级品类,商品价格,省份,商品销量
0,\t271359449904,数码家电,手机,"5,984.69",广东,1000+人付款
1,\t406439219572,数码家电,相机,"5,950.45",广东,2000+人付款
2,\t453132957133,数码家电,相机,"5,931.16",广东,2万+人付款
3,\t655995574060,数码家电,耳机,"5,831.18",广东,2万+人付款
4,\t616431260865,数码家电,手机,"5,809.16",广东,1万+人付款
5,\t552994611034,数码家电,耳机,"5,782.79",广东,500+人付款
6,\t554912900513,数码家电,手机,"5,734.07",广东,2000+人付款
7,\t770192134467,数码家电,相机,"5,689.16",广东,2000+人付款
8,\t472230254988,数码家电,手机,"5,658.84",广东,2000+人付款
9,\t801363237742,数码家电,相机,"5,657.91",广东,200+人付款


浙江或江苏商品数量： 3356


In [37]:
# 检查点4
assert isinstance(guangdong, pd.DataFrame)
assert (guangdong["省份"] == "广东").all()
assert isinstance(guangdong_high_price, pd.DataFrame)
assert (guangdong_high_price["省份"] == "广东").all()
assert (guangdong_high_price["商品价格"] >= 1000).all()
assert guangdong_high_price["商品价格"].is_monotonic_decreasing
assert set(zhejiang_or_jiangsu["省份"].unique()).issubset({"浙江", "江苏"})
print("检查点4通过")

检查点4通过


## 任务5：描述性统计与一级品类汇总

In [38]:
# TODO 1：使用describe查看商品价格摘要，变量名为price_summary
price_summary = df["商品价格"].describe()

# TODO 2：按一级品类统计商品数、平均价格和中位价格
# 按平均价格从高到低排序，变量名为category_summary
category_summary = (
    df.groupby("一级品类")["商品价格"]
      .agg(商品数="count", 平均价格="mean", 中位价格="median")
      .round({"平均价格": 2, "中位价格": 2})
      .sort_values("平均价格", ascending=False)
)

# TODO 3：展示结果
display(price_summary)
display(category_summary)

count   25,000.00
mean       938.26
std      1,017.92
min         11.26
25%        257.39
50%        617.37
75%      1,211.89
max      5,998.78
Name: 商品价格, dtype: float64

,商品数,平均价格,中位价格
一级品类,,,
数码家电,1712,"3,085.53","3,116.12"
钟表珠宝,1656,"1,981.20","1,969.86"
家居生活,1655,"1,527.68","1,494.38"
玩具乐器,1703,"1,259.17","1,269.63"
礼品鲜花,1659,"1,034.35","1,037.23"
运动户外,1684,811.42,801.66
医药健康,1670,791.81,779.60
服饰鞋包,1642,674.52,681.55
母婴用品,1685,666.88,666.25


In [39]:
# 检查点5
assert isinstance(price_summary, pd.Series)
assert isinstance(category_summary, pd.DataFrame)
assert {"商品数", "平均价格", "中位价格"}.issubset(category_summary.columns)
assert category_summary["商品数"].sum() == len(df)
assert category_summary["平均价格"].is_monotonic_decreasing
assert abs(df["商品价格"].mean() - 938.26) < 0.02
print("检查点5通过")

检查点5通过


15 个一级品类样本量高度均衡，数量区间 1624~1712 条，无严重样本失衡：

## 挑战任务：省份—类别分析

In [40]:
# TODO 1：选择两个省份
provinces = ["广东", "江苏"]

# TODO 2：筛选省份并统计商品数、平均价格和中位价格
province_summary = (
    df[df["省份"].isin(provinces)]
      .groupby("省份")["商品价格"]
      .agg(商品数="count", 平均价格="mean", 中位价格="median")
      .round({"平均价格": 2, "中位价格": 2})
      .reindex(provinces)
)

# TODO 3：分别计算两个省份最常见的一级品类
top_categories = (
    df[df["省份"].isin(provinces)]
      .groupby(["省份", "一级品类"]) 
      .size()
      .rename("商品数")
      .reset_index()
      .sort_values(["省份", "商品数"], ascending=[True, False])
      .groupby("省份", as_index=False)
      .first()
)

# TODO 4：展示结果
print("省份汇总：")
display(province_summary)
print("最常见的一级品类：")
display(top_categories)

省份汇总：


,商品数,平均价格,中位价格
省份,,,
广东,2303,911.69,608.62
江苏,1763,936.48,592.41


最常见的一级品类：


,省份,一级品类,商品数
0,广东,数码家电,168
1,江苏,图书音像,130


In [41]:
# 检查点6
assert len(provinces) == 2 and provinces[0] != provinces[1]
assert isinstance(province_summary, pd.DataFrame)
assert set(province_summary.index) == set(provinces)
assert {"商品数", "平均价格", "中位价格"}.issubset(province_summary.columns)
assert isinstance(top_categories, pd.DataFrame)
print("检查点6通过")

检查点6通过


## 输出成果

In [42]:
outputs = {
    "category_summary.csv": category_summary.reset_index(),
    "province_summary.csv": province_summary.reset_index(),
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape
    assert not any(str(col).startswith("Unnamed") for col in reloaded.columns)
    print("已输出：", path.relative_to(ROOT))

已输出： output\day03_analysis\category_summary.csv
已输出： output\day03_analysis\province_summary.csv


## 结论与边界

请至少完成两条结论，每条包含：数据范围、字段与方法、数据结论、结论边界。

### 结论1

> 数据范围：基于 `df` 中全部 25,000 条淘宝商品记录。
> 字段与方法：使用 `商品价格` 的描述性统计和 `category_summary` 中按 `一级品类` 的平均价格排序。
> 数据结论：数码家电、钟表珠宝和家居生活三个一级品类的标价平均值最高，说明这三个品类在本数据集中商品标价整体偏高。
> 结论边界：该结论仅适用于本次样本的标价分布，商品标价不代表实际成交金额，无法直接推断真实交易额或折扣后价格。

### 结论2

> 数据范围：基于 `广东` 和 `江苏` 两个省份的商品子集。
> 字段与方法：使用 `province_summary` 计算 `商品数`、`平均价格` 和 `中位价格`，并用 `top_categories` 查找最常见一级品类。
> 数据结论：江苏样本平均价格略高于广东，但广东的商品数量更多；广东最常见一级品类是数码家电，江苏最常见一级品类是图书音像。
> 结论边界：该结论仅反映所选两个省份与当前数据样本的标价分布，不代表全国市场，也不反映实际成交金额和促销情况。
- [ ] 检查点1～6全部通过；
- [ ] `output/day03_analysis/`包含两个CSV；
- [ ] 结论明确说明商品标价不代表实际成交金额；
- [ ] 已提交并推送到个人GitHub仓库。